# Setup

In [1]:
from getpass import getpass
GITHUB_TOKEN = getpass("")

··········


In [2]:
import pandas as pd
import numpy as np
import requests
import io
import time
import json
import math

GITHUB_RAW = "https://raw.githubusercontent.com/W2NJL/nwr-gap-research/main"
headers = {"Authorization": f"token {GITHUB_TOKEN}"}

def fetch_csv(path):
    r = requests.get(f"{GITHUB_RAW}/{path}", headers=headers)
    r.raise_for_status()
    return pd.read_csv(io.StringIO(r.text))

# --- NWR station data ---
wx = fetch_csv("wx_stations.csv")
wx = wx[wx['country'] == 'USA'].dropna(subset=['latitude', 'longitude']).copy()
wx['longitude'] = -wx['longitude'].abs()
print(f"{len(wx)} US NWR stations loaded.")

# --- Hazard event datasets ---
coastal    = fetch_csv("data/coastal_flood_events.csv")
wildfires  = fetch_csv("data/wildfire_events.csv")
hurricanes = fetch_csv("data/us_hurricane_landfalls.csv")
tornadoes  = fetch_csv("data/tornadoes_f3plus.csv")
print(f"Hazard events: {len(coastal)} coastal flood, {len(wildfires)} wildfire, "
      f"{len(hurricanes)} hurricane landfalls, {len(tornadoes)} tornadoes F3+")

1029 US NWR stations loaded.
Hazard events: 4591 coastal flood, 8440 wildfire, 607 hurricane landfalls, 3278 tornadoes F3+


In [4]:
# --- US cities (population centers)
cities = pd.read_csv("https://raw.githubusercontent.com/CSC-2053-Spring-26-100/lab-12-finding-the-weather-radio-gaps/main/us_cities.csv")
cities = cities[~cities['state_id'].isin(['AK', 'HI', 'PR'])].reset_index(drop=True)
print(f"{len(cities)} contiguous US cities loaded.")

# --- NRI (county-level multi-hazard risk)
nri = pd.read_csv("/content/NRI_Table_Counties.csv")
print(f"{len(nri)} NRI county rows loaded.")

# --- SVI (tract-level social vulnerability)
svi = pd.read_csv("/content/SVI_2022_US.csv")
print(f"{len(svi)} SVI tract rows loaded.")

# --- RUCC (county rurality)
RUCC_URL = "https://ers.usda.gov/sites/default/files/_laserfiche/DataFiles/53251/Ruralurbancontinuumcodes2023.xlsx?v=59358"
import urllib.request
urllib.request.urlretrieve(RUCC_URL, "rucc2023.xlsx")
rucc = pd.read_excel("rucc2023.xlsx")
rucc.columns = rucc.columns.str.strip()
print(f"{len(rucc)} RUCC county rows loaded.")

4834 contiguous US cities loaded.
3232 NRI county rows loaded.
84120 SVI tract rows loaded.
3235 RUCC county rows loaded.


In [5]:
RADIOLAND_BASE = "http://52.151.197.43/search_stream"
COVERAGE_THRESHOLD = 50.0

def query_nwr_coverage(lat, lon, rx_height=10, min_sig_strength=3, verbose=False):
    """
    Query the RadioLand API for NWR stations receivable at (lat, lon).
    Returns a DataFrame sorted by field_strength descending, or None on failure.
    """
    params = {
        "lat": lat, "lon": lon,
        "search_freq": "none", "callsign": "none", "request_type": 1,
        "pi_code": "none", "sig_strength": min_sig_strength, "am_sig_strength": 2,
        "startMiles": "none", "miles": "null", "slogan": "none", "owner": "none",
        "format": "none", "wfo": "none", "rxHeight": rx_height, "mlbTeam": "none",
        "market": "none", "country": "none", "sp": "none", "measurementUnit": "metric",
        "locationName": "", "broadcastBand": "WX", "timeOfDay": "day", "model": "longley_rice",
    }
    try:
        resp = requests.get(RADIOLAND_BASE, params=params, stream=True, timeout=90)
        resp.raise_for_status()
    except requests.RequestException as e:
        if verbose:
            print(f"Request failed: {e}")
        return None

    for raw_line in resp.iter_lines():
        if not raw_line:
            continue
        line = raw_line.decode('utf-8')
        if not line.startswith('data: '):
            continue
        event = json.loads(line[6:])
        if event.get('type') == 'progress' and verbose:
            print(f"  [{event['percentage']:>3}%] {event['message']}", end='\r')
        elif event.get('type') == 'complete':
            if verbose:
                print("  [100%] Done.                                        ")
            result = json.loads(event['data'])
            df = pd.DataFrame(result['data'])
            if df.empty:
                return df
            for col in ['lon', 'transmitter_lon']:
                if col in df.columns:
                    df[col] = -df[col].abs()
            return df.sort_values('field_strength', ascending=False).reset_index(drop=True)
    return None


def get_best_signal(lat, lon, threshold=COVERAGE_THRESHOLD):
    df = query_nwr_coverage(lat, lon)
    if df is None or df.empty:
        return {'best_callsign': None, 'best_field_strength': 0.0,
                'best_distance_km': None, 'stations_above_threshold': 0, 'covered': False}
    best = df.iloc[0]
    return {
        'best_callsign':            best['callsign'],
        'best_field_strength':      float(best['field_strength']),
        'best_distance_km':         float(best.get('distance', float('nan'))),
        'stations_above_threshold': int((df['field_strength'] >= threshold).sum()),
        'covered':                  float(best['field_strength']) >= threshold,
    }

In [6]:
test = get_best_signal(39.8732, -74.6643)
print(test)

{'best_callsign': 'KIH-28', 'best_field_strength': 64.59, 'best_distance_km': 26.86, 'stations_above_threshold': 1, 'covered': True}


# Full City Sweep

In [7]:
import os

OUTFILE = "full_city_sweep.csv"
START_FROM = 0

# Resume support: if a partial file exists, pick up where it left off
if os.path.exists(OUTFILE):
    done_df = pd.read_csv(OUTFILE)
    START_FROM = len(done_df)
    results = done_df.to_dict('records')
    print(f"Resuming from row {START_FROM} ({len(results)} already done)")
else:
    results = []

for i in range(START_FROM, len(cities)):
    row = cities.iloc[i]
    print(f"[{i+1}/{len(cities)}] {row['city']}, {row['state_id']}...", end=' ', flush=True)
    r = get_best_signal(row['lat'], row['lng'])
    r.update({
        'city': row['city'], 'state_id': row['state_id'],
        'lat': row['lat'], 'lng': row['lng'],
        'population': row['population'], 'county_name': row['county_name'],
    })
    results.append(r)
    print("COVERED" if r['covered'] else "GAP", f"({r['best_field_strength']} dB)")

    if (i + 1) % 100 == 0:
        pd.DataFrame(results).to_csv(OUTFILE, index=False)
        print(f"--- checkpoint saved at row {i+1} ---")

    time.sleep(1)

pd.DataFrame(results).to_csv(OUTFILE, index=False)
print(f"\nDone. Saved {len(results)} rows to {OUTFILE}")

Resuming from row 4834 (4834 already done)

Done. Saved 4834 rows to full_city_sweep.csv


# Stage 2: Build rural grid with already included cities filtered out.

In [8]:
GRID_SPACING = 0.5
EXCLUSION_RADIUS_MI = 30

lat_vals = np.arange(25.0, 49.5, GRID_SPACING)
lon_vals = np.arange(-125.0, -66.5, GRID_SPACING)
grid_lat, grid_lon = np.meshgrid(lat_vals, lon_vals)
grid_points = pd.DataFrame({'lat': grid_lat.ravel(), 'lon': grid_lon.ravel()})
print(f"Raw grid: {len(grid_points)} points")

def min_dist_to_cities_miles(glat, glon, city_lats, city_lons):
    R = 3958.8
    p1 = np.radians(glat)
    p2 = np.radians(city_lats)
    dphi = np.radians(city_lats - glat)
    dlambda = np.radians(city_lons - glon)
    a = np.sin(dphi/2)**2 + np.cos(p1)*np.cos(p2)*np.sin(dlambda/2)**2
    d = 2*R*np.arcsin(np.sqrt(a))
    return d.min()

city_lats = cities['lat'].values
city_lngs = cities['lng'].values

grid_points['min_dist_mi'] = grid_points.apply(
    lambda r: min_dist_to_cities_miles(r['lat'], r['lon'], city_lats, city_lngs), axis=1
)

rural_grid = grid_points[grid_points['min_dist_mi'] > EXCLUSION_RADIUS_MI].reset_index(drop=True)
print(f"Filtered rural grid: {len(rural_grid)} points (removed {len(grid_points)-len(rural_grid)} near-city points)")

est_hours = len(rural_grid) * 4.4 / 3600
print(f"Estimated time: {est_hours:.1f} hours")

Raw grid: 5733 points
Filtered rural grid: 3376 points (removed 2357 near-city points)
Estimated time: 4.1 hours


In [9]:
GRID_OUTFILE = "rural_grid_sweep.csv"
START_FROM = 0

if os.path.exists(GRID_OUTFILE):
    done_df = pd.read_csv(GRID_OUTFILE)
    START_FROM = len(done_df)
    grid_results = done_df.to_dict('records')
    print(f"Resuming from row {START_FROM} ({len(grid_results)} already done)")
else:
    grid_results = []

for i in range(START_FROM, len(rural_grid)):
    row = rural_grid.iloc[i]
    print(f"[{i+1}/{len(rural_grid)}] ({row['lat']:.2f}, {row['lon']:.2f})...", end=' ', flush=True)
    r = get_best_signal(row['lat'], row['lon'])
    r.update({'lat': row['lat'], 'lon': row['lon'], 'min_dist_to_city_mi': row['min_dist_mi']})
    grid_results.append(r)
    print("COVERED" if r['covered'] else "GAP", f"({r['best_field_strength']} dB)")

    if (i + 1) % 100 == 0:
        pd.DataFrame(grid_results).to_csv(GRID_OUTFILE, index=False)
        print(f"--- checkpoint saved at row {i+1} ---")

    time.sleep(1)

pd.DataFrame(grid_results).to_csv(GRID_OUTFILE, index=False)
print(f"\nDone. Saved {len(grid_results)} rows to {GRID_OUTFILE}")

Resuming from row 3376 (3376 already done)

Done. Saved 3376 rows to rural_grid_sweep.csv


# Joins and Scores